# 03 - Data Preparation, Rotulagem e Features

Explica o dataset modelável, leakage control e principais famílias de features.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0]
feat_path = ROOT / 'data' / 'processed' / 'features' / 'features_dataset.parquet'
df = pd.read_parquet(feat_path)
print('Shape features:', df.shape)
df.head(2)

Shape features: (377907, 57)


,Id,Inicio,Fim,Tag,Classe,next_critical_event_time,tte_horas,target_4h,hora_do_dia,dia_da_semana,...,Tag_freq,Operador_freq,Classe_target_enc,Frota_793-D 2S,Frota_793-D 3S,Frota_793-D 4S,Frota_793-D 5S,Frota_LeTourneau L 1850,Tipo_Caminhao,Tipo_Escavadeira
0,23130822,2025-01-01 03:00:00+00:00,2025-01-01 04:00:00+00:00,CA0000,Parado,NaT,NaN,0,3,2,...,0.008812,0.0,0.357143,False,False,False,True,False,True,False
1,23132927,2025-01-01 04:00:00+00:00,2025-01-01 05:00:00+00:00,CA0000,Parado,NaT,NaN,0,4,2,...,0.008812,0.0,0.274510,False,False,False,True,False,True,False


In [2]:
# Colunas críticas para vazamento (devem sair antes do treino)
leakage_candidates = ['Id', 'Inicio', 'Fim', 'Tag', 'Classe', 'next_critical_event_time', 'tte_horas', 'target_4h']
present = [c for c in leakage_candidates if c in df.columns]
present

['Id',
 'Inicio',
 'Fim',
 'Tag',
 'Classe',
 'next_critical_event_time',
 'tte_horas',
 'target_4h']

In [3]:
# Resumo de tipos de variáveis
dtype_summary = df.dtypes.astype(str).value_counts().rename_axis('dtype').reset_index(name='qtd')
dtype_summary

,dtype,qtd
0,float64,31
1,int64,8
2,bool,7
3,int32,5
4,"datetime64[ns, UTC]",3
5,object,3


In [4]:
# Exemplo de famílias de features esperadas
familias = {
    'temporais': [c for c in df.columns if 'hora' in c or 'dia' in c or 'mes' in c],
    'rolling_alerta': [c for c in df.columns if 'alerta' in c or 'n_alertas' in c],
    'rolling_ciclo': [c for c in df.columns if 'duracao' in c or 'n_ciclos' in c],
    'encoding': [c for c in df.columns if c.endswith('_freq') or c.endswith('_target_enc') or c.startswith('Frota_') or c.startswith('Tipo_')],
}
{k: len(v) for k, v in familias.items()}

{'temporais': 14, 'rolling_alerta': 9, 'rolling_ciclo': 13, 'encoding': 10}